In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_orders = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/seo_orders/seo_orders.csv")
)

print("Rows:", df_orders.count())
print("Columns:", len(df_orders.columns))

display(df_orders.limit(10))

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 3, Finished, Available, Finished, False)

Rows: 10002
Columns: 9


SynapseWidget(Synapse.DataFrame, 5ce0fdc3-49c6-4cdb-aabc-255ebe66e685)

In [2]:
print("Order status distribution:")

display(
    df_orders
    .groupBy("order_status")
    .count()
    .orderBy("order_status")
)

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 4, Finished, Available, Finished, False)

Order status distribution:


SynapseWidget(Synapse.DataFrame, 8c42b685-bfdc-4f12-9db7-f7c493aeaa7e)

In [3]:
print("Order amount statistics:")

df_orders.select(
    F.min("order_amount").alias("min_amount"),
    F.max("order_amount").alias("max_amount"),
    F.sum(
        F.when(F.col("order_amount") < 0, 1).otherwise(0)
    ).alias("negative_amount_records"),
    F.sum(
        F.when(F.col("order_amount").isNull(), 1).otherwise(0)
    ).alias("null_amount_records")
).show()

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 5, Finished, Available, Finished, False)

Order amount statistics:
+----------+----------+-----------------------+-------------------+
|min_amount|max_amount|negative_amount_records|null_amount_records|
+----------+----------+-----------------------+-------------------+
|    -100.0|   8980.44|                      1|                  0|
+----------+----------+-----------------------+-------------------+



In [4]:
print("Duplicate order IDs:")

duplicate_orders = (
    df_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Number of duplicated order IDs:", duplicate_orders.count())

display(
    duplicate_orders
    .orderBy(F.col("count").desc())
    .limit(10)
)

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 6, Finished, Available, Finished, False)

Duplicate order IDs:
Number of duplicated order IDs: 2


SynapseWidget(Synapse.DataFrame, 7e9daf9f-773b-4437-b787-a61415e902f6)

In [5]:
window_orders = Window.partitionBy(
    "order_id"
).orderBy(
    F.col("updated_timestamp").desc()
)

df_orders_clean = (
    df_orders
    .withColumn("_rn", F.row_number().over(window_orders))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print("Rows after deduplication:", df_orders_clean.count())

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 7, Finished, Available, Finished, False)

Rows after deduplication: 10000


In [6]:
df_orders_clean = df_orders_clean.withColumn(
    "country_code",
    F.when(
        F.upper(F.trim(F.col("country_code"))).isin(
            "USA", "UNITED STATES", "US"
        ),
        F.lit("US")
    )
    .otherwise(
        F.upper(F.trim(F.col("country_code")))
    )
)

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 8, Finished, Available, Finished, False)

In [7]:
print("Order status distribution after deduplication:")

display(
    df_orders_clean
    .groupBy("order_status")
    .count()
    .orderBy("order_status")
)

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 9, Finished, Available, Finished, False)

Order status distribution after deduplication:


SynapseWidget(Synapse.DataFrame, ec1ecf93-a103-4bb7-96f1-b1ec49d01f4c)

In [8]:
print("Negative order amounts:")

print(
    df_orders_clean
    .filter(F.col("order_amount") < 0)
    .count()
)

display(
    df_orders_clean
    .filter(F.col("order_amount") < 0)
    .limit(10)
)

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 10, Finished, Available, Finished, False)

Negative order amounts:
1


SynapseWidget(Synapse.DataFrame, dfaf18da-1e83-4a59-8f12-5493bb585643)

In [9]:
df_orders_clean = df_orders_clean.filter(
    (F.upper(F.trim(F.col("order_status"))) == "COMPLETED") &
    (F.col("order_amount") > 0)
)

print("Rows after business validation:", df_orders_clean.count())

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 11, Finished, Available, Finished, False)

Rows after business validation: 4987


In [10]:
print("Remaining order statuses:")

display(
    df_orders_clean
    .groupBy("order_status")
    .count()
)

print("Negative amounts remaining:")

print(
    df_orders_clean
    .filter(F.col("order_amount") <= 0)
    .count()
)

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 12, Finished, Available, Finished, False)

Remaining order statuses:


SynapseWidget(Synapse.DataFrame, e6a81a2f-ac59-452a-8d58-5fb300d3b7e2)

Negative amounts remaining:
0


In [11]:
print("Final row count:", df_orders_clean.count())
print("Final column count:", len(df_orders_clean.columns))

print("\nDuplicate order IDs:")
print(
    df_orders_clean
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nInvalid order statuses:")
print(
    df_orders_clean
    .filter(
        F.upper(F.trim(F.col("order_status"))) != "COMPLETED"
    )
    .count()
)

print("\nInvalid order amounts:")
print(
    df_orders_clean
    .filter(F.col("order_amount") <= 0)
    .count()
)

print("\nNull order IDs:")
print(
    df_orders_clean
    .filter(F.col("order_id").isNull())
    .count()
)

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 13, Finished, Available, Finished, False)

Final row count: 4987
Final column count: 9

Duplicate order IDs:
0

Invalid order statuses:
0

Invalid order amounts:
0

Null order IDs:
0


In [12]:
df_orders_clean.select(
    F.countDistinct("order_id").alias("completed_orders"),
    F.sum("order_amount").alias("total_revenue"),
    F.avg("order_amount").alias("average_order_value")
).show()

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 14, Finished, Available, Finished, False)

+----------------+-----------------+-------------------+
|completed_orders|    total_revenue|average_order_value|
+----------------+-----------------+-------------------+
|            4987|4830681.510000006|  968.6548044916797|
+----------------+-----------------+-------------------+



In [13]:
(
    df_orders_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_seo_orders")
)

print("Silver fact_seo_orders table created successfully.")

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 15, Finished, Available, Finished, False)

Silver fact_seo_orders table created successfully.


In [14]:
df_silver_orders = spark.read.table("fact_seo_orders")

print("Silver orders rows:", df_silver_orders.count())
print("Silver orders columns:", len(df_silver_orders.columns))

display(df_silver_orders.limit(10))

StatementMeta(, 5b2084cb-e4ba-4b81-a99a-99ebfdd7d152, 16, Finished, Available, Finished, False)

Silver orders rows: 4987
Silver orders columns: 9


SynapseWidget(Synapse.DataFrame, 857441ff-bcdb-4e64-bf42-51048a3f043f)